# steps_multi_time_fusion 算法使用教程

本 notebook 面向算法使用者，说明 `steps_multi_time_fusion` 的用途、目录结构、公开插件和典型调用流程。

`steps_multi_time_fusion` 包含三个主要能力：

- `StepsNoisePlugin`：训练非参数滤波器并生成 AR(2) 随机噪声。
- `ClimatologicalSkillPlugin`：计算逐日技巧评分，并汇总气候态技巧评分。
- `StepsBlendingPlugin`：使用 STEPS 级联分解和权重进行 nowcast/NWP 融合。

真实业务运行需要真实降水场、历史样本、技巧评分文件和业务输出路径。本教程默认只执行轻量合成数据示例。

## 0. 打开和运行环境

建议使用 `pytorch` 环境作为 notebook kernel。

如果在 PyCharm Community 中只能预览 notebook，可以在命令行运行：

```powershell
conda activate pytorch
cd D:/nimm-file/cli_code/steps_multi_time_fusion/nbs
jupyter lab
```

然后在浏览器中打开 `steps_tutorial.ipynb`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "nbs":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"D:/nimm-file/cli_code/steps_multi_time_fusion")

WORKSPACE_ROOT = PROJECT_ROOT.parent
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

print("算法目录:", PROJECT_ROOT)
print("工作区目录:", WORKSPACE_ROOT)

## 1. 目录结构

| 目录/文件 | 说明 |
| --- | --- |
| `src/noise_plugin.py` | 噪声训练和生成插件 |
| `src/climatological_skill_plugin.py` | 技巧评分插件 |
| `src/steps_blending_plugin.py` | STEPS 融合插件 |
| `cli/noise.py` | 噪声生成命令行入口 |
| `cli/climatological_skill.py` | 气候态技巧评分汇总命令行入口 |
| `cli/blending.py` | npy 双源融合验证入口 |
| `test/` | 轻量单元测试 |

In [ ]:
for path in [
    PROJECT_ROOT / "src" / "noise_plugin.py",
    PROJECT_ROOT / "src" / "climatological_skill_plugin.py",
    PROJECT_ROOT / "src" / "steps_blending_plugin.py",
    PROJECT_ROOT / "cli" / "blending.py",
]:
    print(path)

## 2. 公开插件

插件统一从 `steps_multi_time_fusion.src` 导入：

```python
from steps_multi_time_fusion.src import StepsNoisePlugin, ClimatologicalSkillPlugin, StepsBlendingPlugin
```

三个插件各自负责 STEPS 流程中的一个环节，可以单独使用，也可以组合成完整流程。

In [ ]:
from steps_multi_time_fusion.src import StepsNoisePlugin, ClimatologicalSkillPlugin, StepsBlendingPlugin

skill_plugin = ClimatologicalSkillPlugin(n_cascade_levels=3)
blend_plugin = StepsBlendingPlugin(n_cascade_levels=4)

print(skill_plugin)
print(blend_plugin)

## 3. 气候态技巧评分 ClimatologicalSkillPlugin

该插件用于评估 nowcast 与观测在不同提前量和级联尺度上的相关技巧。

典型输入：

- `nowcast_stack`：形状为 `(lead_time, y, x)` 的预报序列。
- `obs`：形状为 `(y, x)` 的观测场。

输出：

- `skill`：逐提前量、逐级联层的技巧矩阵。
- `raw`：原始相关矩阵。
- `climatological_skill`：多个 daily skill 汇总后的气候态技巧评分。

In [ ]:
import numpy as np

nowcast_stack = np.ones((3, 8, 8), dtype=np.float32)
obs = np.ones((8, 8), dtype=np.float32)

skill, raw = skill_plugin.process(nowcast_stack, obs)
clim_skill = skill_plugin.climatological_skill([skill])

print("daily skill shape:", skill.shape)
print("raw shape:", raw.shape)
print("climatological skill shape:", clim_skill.shape)

## 4. 噪声训练和生成 StepsNoisePlugin

该插件有两个动作：

- `train(fields)`：根据历史降水场训练非参数滤波器，输出 `nonparam_filters_*.npy`。
- `generate(filter_path)`：基于滤波器生成 AR(2) 噪声集合，输出 `ar_noise_*.npy`。

常用参数：

| 参数 | 说明 |
| --- | --- |
| `output_dir` | 滤波器和噪声输出目录 |
| `issue_time` | 起报时间标识 |
| `n_levels` | 级联层数 |
| `n_ens_members` | 集合成员数 |
| `timesteps` | 噪声时间步数 |

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as tmp_dir:
    fields = [np.ones((8, 8), dtype=np.float32) * value for value in (1.0, 2.0, 3.0)]
    noise_plugin = StepsNoisePlugin(
        output_dir=tmp_dir,
        issue_time="202601010000",
        n_levels=3,
        n_ens_members=2,
        timesteps=2,
    )
    filter_path = noise_plugin.train(fields)
    noise_path = noise_plugin.generate(filter_path)
    print("滤波器:", filter_path)
    print("噪声文件:", noise_path)
    print("滤波器 shape:", np.load(filter_path).shape)
    print("噪声 shape:", np.load(noise_path).shape)

## 5. STEPS 级联融合 StepsBlendingPlugin

该插件对单个提前量的二维 nowcast 和 NWP 降水场进行级联融合。

典型输入：

- `nowcast`：二维降水场，形状 `(y, x)`。
- `nwp`：二维模式降水场，形状 `(y, x)`。
- `lead_index`：10 分钟步长索引，`1` 表示 +10min。
- `noise`：可选二维随机噪声场。

可选使用 `climatological_skill_path` 指向气候态技巧评分文件。

In [ ]:
nowcast = np.ones((16, 16), dtype=np.float64) * 2.0
nwp = np.ones((16, 16), dtype=np.float64)

result = blend_plugin.process(nowcast, nwp, lead_index=1)

print("融合结果 shape:", result.shape)
print("最小值/最大值:", float(result.min()), float(result.max()))

## 6. 真实业务组合流程模板

真实业务中通常按以下顺序组织：

1. 准备历史降水场，训练非参数滤波器。
2. 生成 AR(2) 噪声成员。
3. 根据历史 daily skill 汇总气候态技巧评分。
4. 对每个提前量逐帧调用 `StepsBlendingPlugin.process()`。

下面是调用模板，确认真实数据路径后再取消注释执行。

In [ ]:
# 真实业务调用模板：确认真实数据和输出目录后再取消注释执行。
# noise_plugin = StepsNoisePlugin(
#     output_dir="/path/to/output",
#     issue_time="202601010000",
#     n_levels=6,
#     n_ens_members=20,
#     timesteps=18,
# )
# filter_path = noise_plugin.train(history_fields)
# noise_path = noise_plugin.generate(filter_path)
#
# skill_plugin = ClimatologicalSkillPlugin(n_cascade_levels=8)
# skill, raw = skill_plugin.process(nowcast_stack, obs)
# clim = skill_plugin.climatological_skill([skill_day1, skill_day2, skill_day3])
#
# blend_plugin = StepsBlendingPlugin(
#     n_cascade_levels=8,
#     use_climatological_skill=True,
#     climatological_skill_path="/path/to/climatological_skill.npy",
# )
# blended = blend_plugin.process(nowcast_2d, nwp_2d, lead_index=1, noise=noise_2d)

## 7. 命令行等价调用

噪声生成：

```powershell
cd D:/nimm-file/cli_code
python -m steps_multi_time_fusion.cli.noise generate --output-dir D:/tmp/steps --issue-time 202601010000
```

气候态技巧评分汇总：

```powershell
python -m steps_multi_time_fusion.cli.climatological_skill --input-dir D:/tmp/daily_skill --output D:/tmp/climatological_skill.npy
```

二维 npy 融合验证：

```powershell
python -m steps_multi_time_fusion.cli.blending --nowcast nowcast.npy --nwp nwp.npy --output blended.npy
```

## 8. 使用前检查清单

- 输入降水场维度一致，单位和缺测处理规则一致。
- 训练噪声滤波器时历史样本数量足够。
- `n_cascade_levels` 在噪声、技巧评分和融合流程中保持一致。
- 若使用气候态技巧评分，确认 `climatological_skill.npy` 的形状与级联层数匹配。
- 真实业务批处理前，先用小范围数据验证输出。